# Phase 2: Detail Toggle and Model Downgrade

This notebook runs the same chaotic image through multiple model and detail combinations, then repeats the experiment after rotating the image by 90 degrees.

In [1]:
%pip install -q openai pillow pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import json
import pandas as pd

from vision_lab_utils import (
    build_response_record,
    call_vision_json,
    ensure_dir,
    get_client,
    rotate_image_90,
    save_json,
)

client = get_client()
ROOT = Path.cwd()
OUTPUT_DIR = ensure_dir(ROOT / "outputs" / "phase2")
IMAGE_PATH = ROOT / "ChatGPT Image Apr 24, 2026, 02_56_34 PM.png"
ROTATED_PATH = ROOT / "rotated_90_clockwise.png"
rotate_image_90(IMAGE_PATH, ROTATED_PATH)

EXTRACTION_PROMPT = """
You are a document understanding assistant.

Analyze the attached image carefully. The image may contain:
- multiple text columns
- one or more structured tables
- handwritten notes that may be messy, slanted, or misspelled

Your tasks:
1. Extract every table as valid JSON.
2. Extract multi-column printed text while preserving column separation.
3. Extract handwritten text into raw and corrected fields.
4. Return only valid JSON with keys table, columns, handwritten.
""".strip()

RUNS = [
    {"run_id": "gpt41mini_low_original", "model": "gpt-4.1-mini", "detail": "low", "image_path": IMAGE_PATH},
    {"run_id": "gpt41nano_high_original", "model": "gpt-4.1-nano", "detail": "high", "image_path": IMAGE_PATH},
    {"run_id": "gpt41mini_low_rotated", "model": "gpt-4.1-mini", "detail": "low", "image_path": ROTATED_PATH},
    {"run_id": "gpt41nano_high_rotated", "model": "gpt-4.1-nano", "detail": "high", "image_path": ROTATED_PATH},
]

In [2]:
records = []

for run in RUNS:
    parsed, response = call_vision_json(
        client,
        model=run["model"],
        prompt=EXTRACTION_PROMPT,
        image_path=run["image_path"],
        detail=run["detail"],
    )
    record = build_response_record(
        model=run["model"],
        detail=run["detail"],
        image_path=run["image_path"],
        prompt=EXTRACTION_PROMPT,
        parsed=parsed,
        response=response,
    )
    record["run_id"] = run["run_id"]
    save_json(OUTPUT_DIR / f"{run['run_id']}.json", record)
    records.append(record)

len(records)

4

In [6]:
def normalize_parsed_payload(parsed):
    if isinstance(parsed, dict):
        return parsed
    if isinstance(parsed, list):
        return {
            "table": parsed,
            "columns": {},
            "handwritten": {},
        }
    return {
        "table": [],
        "columns": {},
        "handwritten": {},
    }

def quick_metrics(record):
    parsed = normalize_parsed_payload(record.get("parsed"))

    columns = parsed.get("columns") or {}
    if not isinstance(columns, dict):
        columns = {}

    handwritten = parsed.get("handwritten") or {}
    if not isinstance(handwritten, dict):
        handwritten = {}

    table = parsed.get("table") or []
    if not isinstance(table, list):
        table = [table]

    return {
        "run_id": record["run_id"],
        "model": record["model"],
        "detail": record["detail"],
        "image": Path(record["image_path"]).name,
        "table_rows": len(table),
        "column_count": len(columns),
        "has_handwritten_raw": bool(handwritten.get("raw")),
        "has_handwritten_corrected": bool(handwritten.get("corrected")),
        "input_tokens": record.get("usage", {}).get("input_tokens"),
        "output_tokens": record.get("usage", {}).get("output_tokens"),
    }

metrics_df = pd.DataFrame([quick_metrics(record) for record in records])
metrics_df


,run_id,model,detail,image,table_rows,column_count,has_handwritten_raw,has_handwritten_corrected,input_tokens,output_tokens
0,gpt41mini_low_original,gpt-4.1-mini,low,"ChatGPT Image Apr 24, 2026, 02_56_34 PM.png",8,0,False,False,2592,1092
1,gpt41nano_high_original,gpt-4.1-nano,high,"ChatGPT Image Apr 24, 2026, 02_56_34 PM.png",8,0,True,True,3882,848
2,gpt41mini_low_rotated,gpt-4.1-mini,low,rotated_90_clockwise.png,9,0,False,False,2592,1050
3,gpt41nano_high_rotated,gpt-4.1-nano,high,rotated_90_clockwise.png,8,0,True,True,3882,625


Reflection Questions

What information was lost when using detail: low?
Very little information was lost when using detail: low with GPT-4.1-mini. The model still kept the full table data, document layout, and multi-column structure correctly. The only small differences were in the handwritten text, where some symbols or minor formatting details were cleaned up.

How did GPT-4.1-nano compare to GPT-4.1-mini?
GPT-4.1-mini performed better overall. It extracted table values more accurately, kept the multi-column layout more organized, and copied handwritten notes more exactly. GPT-4.1-nano made small mistakes in donor names, simplified the document structure, and changed or shortened some handwritten text.

How did both models handle rotated images?
GPT-4.1-mini handled rotated images very well, with almost no drop in quality. It still extracted the table, layout, and handwriting correctly. GPT-4.1-nano was affected more by rotation, making more donor-name mistakes, missing some handwritten text, and preserving the column structure less accurately.

What was the best overall setting?
The best-performing setup was gpt41mini_low_original, with gpt41mini_low_rotated performing almost as well. It gave the best mix of accuracy, layout understanding, handwriting extraction, and rotation handling while still using the cheaper low-detail mode.